# CenpAI detector comparison (Phase 5, step 6)

One parameterised notebook for **every run** in `docs/experiments/detector_matrix.md`: pick the settings in cell 2, run all cells, and the run saves itself to Drive. Change the settings and run again for the next row of the matrix.

What it does, in order: unzips dataset v2 to local disk, builds the training list (sources on or off, optional training-only synthetic folders), checks for leaks, trains (resuming from `last.pt` after a disconnect), picks `conf` and `mask_conf` on **validation**, exports ONNX at 640 and 416 with a `MODEL.json`, and records the ONNX output shape with each model.

Rules (`docs/protocols/data_and_phone_test.md`):
- Every choice is made on validation: **PH BB v5 val** and **Open Images val**, reported separately and together (the `val_<source>.txt` lists that `ml.dataset` writes).
- The test splits are touched only in the last cell, marked *locked comparison, run once*. The family-car test videos are never read here.
- Nothing secret is needed or stored here. Every number it prints is measured in the run; nothing is typed in.
- Models started from v0 (`START` = `v0`) have seen part of PH BB v5 val and test, so their PH BB columns carry a caveat and Open Images val is the clean column (see the matrix, "Leak warning").

**Before running** (Colab, GPU runtime = T4): dataset v2 must be at `MyDrive/CenpAI/data/processed/dataset-v2.zip` (built by `notebooks/build_dataset_v2.ipynb`). Synthetic folders, if used, come from `notebooks/make_synthetic.ipynb` and sit under `MyDrive/CenpAI/data/synthetic/`.

In [ ]:
# 1. Setup
!pip -q install ultralytics onnx onnxslim onnxruntime pyyaml
import os, sys, json, shutil, zipfile, subprocess, hashlib, time, random
from pathlib import Path
import numpy as np, torch, ultralytics, onnxruntime
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| onnxruntime", onnxruntime.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to a GPU)")

from google.colab import drive
drive.mount("/content/drive")

REPO_BRANCH = "feat/dataset-v2"  #@param {type:"string"}
REPO = Path("/content/cenpai")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "https://github.com/RedGla/cenpai", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from ml import dataset as ds
assert hasattr(ds, "leak_report"), "ml.dataset on this branch has no leak_report: set REPO_BRANCH to a branch that has dataset v2 (feat/dataset-v2 until it is merged)"

ROOT = Path("/content/drive/MyDrive/CenpAI")
V2_ZIP = ROOT / "data" / "processed" / "dataset-v2.zip"
SYN_DIR = ROOT / "data" / "synthetic"
RUNS = ROOT / "runs" / "compare"           # one folder per RUN_ID, on Drive
RELEASE = ROOT / "release" / "compare"
for p in (RUNS, RELEASE):
    p.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/work")
V2 = WORK / "dataset-v2"

## 2. Settings for this run

One row of the matrix = one setting of this cell. Example rows (all from `docs/experiments/detector_matrix.md`):

| Run | RUN_ID | START | EPOCHS | AUG | SOURCES / EXCLUDE | SYNTHETIC |
|---|---|---|---|---|---|---|
| R1 | `R1` | `v0` | 50 | strong | all | none |
| R1c | `R1c` | `yolov8n.pt` | 50 | strong | all | none |
| B0 | `B0` | `yolov8n.pt` | 30 | strong | all | none |
| B1 | `B1` | `yolov8n.pt` | 30 | strong | `SOURCES` = the three PH sources | none |
| B2 | `B2` | `yolov8n.pt` | 30 | strong | `EXCLUDE_SOURCES` = `open-images-v7-plates` | none |
| B7 | `B7` | `yolov8n.pt` | 30 | default | all | none |
| S1 | `S1` | `yolov8n.pt` | 30 | strong | all | `variants` |
| C1 seed 1 | `C1-s1` | `yolov8n.pt` | 50 | strong | best mix from B | best mix from S |

`RUN_ID` must be new for every run (it names the Drive folder). To continue a run after a disconnect, keep the same settings and `RUN_ID`, and run all cells again.

In [ ]:
# Settings (change these, then Runtime > Run all)
RUN_ID = "R1c"                       #@param {type:"string"}
START = "yolov8n.pt"                 #@param {type:"string"}
# START: a model name Ultralytics downloads (yolov8n.pt, yolov8s.pt, yolo11n.pt, yolo11s.pt, yolo26n.pt),
# or "v0" for the demo detector at MyDrive/CenpAI/release/yolov8n/best.pt, or a full path to any .pt on Drive.
SEED = 0                             #@param {type:"integer"}
EPOCHS = 50                          #@param {type:"integer"}
IMGSZ = 640                          #@param [320, 416, 640] {type:"raw"}
BATCH = 16                           #@param {type:"integer"}
AUG = "strong"                       #@param ["strong", "default"]
PATIENCE = -1                        #@param {type:"integer"}
# PATIENCE -1 = early stopping off (patience = EPOCHS), as blocks C and D require.
SOURCES = "all"                      #@param {type:"string"}
# SOURCES: "all", or a comma list of source names to keep (training images only; val and test are never filtered).
EXCLUDE_SOURCES = ""                 #@param {type:"string"}
# EXCLUDE_SOURCES: comma list of source names to drop from training, e.g. "ccpd2019-sample,keyboard-rover-v4".
SYNTHETIC = ""                       #@param {type:"string"}
# SYNTHETIC: comma list of folders under MyDrive/CenpAI/data/synthetic/ to add to training: variants, copypaste, screen, rendered.
CAPS = ""                            #@param {type:"string"}
# CAPS: cap the training images of some sources, "source:N,source:N" (a fixed random sample, the same for every seed and run).
# Smaller training set = fewer GPU hours. Example: "open-images-v7-plates:1500,ccpd2019-sample:400,keyboard-rover-v4:300".
PH_OVERSAMPLE = 1                    #@param {type:"integer"}
# PH_OVERSAMPLE: list the PH BB v5 training images this many times (use 2 if internet data is more than half of training).
PH_SOURCE = "license-plate-detection-ph-bb-v5"
OI_SOURCE = "open-images-v7-plates"

STRONG_AUG = dict(hsv_v=0.6, hsv_s=0.7, degrees=10, perspective=0.0008, scale=0.6, translate=0.2,
                  mosaic=1.0, mixup=0.1, close_mosaic=10)
AUG_ARGS = STRONG_AUG if AUG == "strong" else {}          # "default" = Ultralytics defaults
START_PATH = str(ROOT / "release" / "yolov8n" / "best.pt") if START == "v0" else START
STARTED_FROM_V0 = START == "v0"     # has seen part of PH BB val and test (a path to v0's best.pt counts too: set START to "v0")
PATIENCE_USED = EPOCHS if PATIENCE < 0 else PATIENCE
RUN_DIR = RUNS / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SETTINGS = dict(run_id=RUN_ID, start=START, seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, aug=AUG, aug_args=AUG_ARGS,
                patience=PATIENCE_USED, sources=SOURCES, exclude_sources=EXCLUDE_SOURCES, caps=CAPS, synthetic=SYNTHETIC,
                ph_oversample=PH_OVERSAMPLE, repo_branch=REPO_BRANCH, started_from_v0=STARTED_FROM_V0)
(RUN_DIR / "settings.json").write_text(json.dumps(SETTINGS, indent=2) + "\n")
print(json.dumps(SETTINGS, indent=2))
if STARTED_FROM_V0:
    print("\nCAVEAT: this model starts from v0, which trained on Roboflow's PH BB v5 train split. Part of v2's PH BB val and test were in that split, "
          "so PH BB columns are optimistic. Judge it on Open Images val.")

## 3. Dataset: unzip to local disk, then build the training list

Drive keeps one zip; the files live on local disk (thousands of small files are slow through the Drive mount). Training images are named `<source>__<sha>.<ext>`, so a source is switched on or off by its prefix. Validation is never filtered.

In [ ]:
IMG_EXT = ds.IMAGE_EXTS
if not (V2 / "manifest.csv").is_file():
    assert V2_ZIP.is_file(), f"{V2_ZIP} not found: run notebooks/build_dataset_v2.ipynb first"
    shutil.copy(V2_ZIP, "/content/v2.zip")
    with zipfile.ZipFile("/content/v2.zip") as z:
        z.extractall(V2)
    os.remove("/content/v2.zip")
report = json.loads((V2 / "report.json").read_text())
print("dataset", report["version"], "| train", report["splits"]["train"], "| val", report["splits"]["val"], "| test", report["splits"]["test"])

def images_in(d):
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMG_EXT)

def label_of(img):
    return Path(str(img).replace("/images/", "/labels/").replace("\\images\\", "\\labels\\")).with_suffix(".txt")

def read_list(name):
    # val_<source>.txt lines look like ./images/val/<file>, relative to the dataset root
    f = V2 / name
    return [V2 / l.strip()[2:] for l in f.read_text().splitlines() if l.strip()] if f.is_file() else []

def src_of(p):
    return Path(p).name.split("__")[0]

all_sources = sorted({src_of(p) for p in images_in(V2 / "images" / "train")} | set(report["sources"]))
drop = {s.strip() for s in EXCLUDE_SOURCES.split(",") if s.strip()}
keep = set(all_sources) if SOURCES.strip() == "all" else {s.strip() for s in SOURCES.split(",") if s.strip()}
unknown = (keep | drop) - set(all_sources)
assert not unknown, f"unknown source names {sorted(unknown)}; known: {all_sources}"
keep -= drop

train_imgs = [p for p in images_in(V2 / "images" / "train") if src_of(p) in keep]
caps = {}
for item in CAPS.split(","):
    if item.strip():
        k, n = item.split(":")
        caps[k.strip()] = int(n)
assert set(caps) <= set(all_sources), f"unknown source in CAPS: {sorted(set(caps) - set(all_sources))}"
capped = []
for s in sorted({src_of(p) for p in train_imgs}):
    grp = [p for p in train_imgs if src_of(p) == s]
    if s in caps and len(grp) > caps[s]:
        grp = random.Random(f"caps:{s}").sample(grp, caps[s])     # fixed: the same subset in every run
    capped += grp
train_imgs = sorted(capped)
train_list = []
for p in train_imgs:
    train_list += [p] * (PH_OVERSAMPLE if src_of(p) == PH_SOURCE else 1)
syn_folders = [SYN_DIR / s.strip() for s in SYNTHETIC.split(",") if s.strip()]
for d in syn_folders:
    assert (d / "images").is_dir() and (d / "labels").is_dir(), f"{d} has no images/ and labels/ (run make_synthetic.ipynb)"
syn_count = {d.name: len(images_in(d / "images")) for d in syn_folders}

DATA_DIR = WORK / "runs-data" / RUN_ID           # same path every session, so resume finds its data yaml
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "train.txt").write_text("".join(f"{p.as_posix()}\n" for p in train_list))

per_src = {}
for p in train_imgs:
    per_src[src_of(p)] = per_src.get(src_of(p), 0) + 1
n_syn = sum(syn_count.values())
n_train_total = len(train_list) + n_syn
ph_n = per_src.get(PH_SOURCE, 0) * PH_OVERSAMPLE
print("training sources kept:", {k: per_src.get(k, 0) for k in sorted(keep)})
print("synthetic folders:", syn_count)
print(f"training list: {len(train_list)} real lines (PH BB listed x{PH_OVERSAMPLE}) + {n_syn} synthetic = {n_train_total}")
if n_syn:
    print(f"synthetic share of training: {n_syn / n_train_total:.1%} (protocol: keep it to at most about a third to a half)")
print(f"PH BB share of real training lines: {ph_n / max(1, len(train_list)):.1%}")
# Training data: the list file plus each synthetic folder's images folder. Ultralytics finds labels next to /images/.
TRAIN_ENTRIES = [str(DATA_DIR / "train.txt")] + [str(d / "images") for d in syn_folders]

# Validation sets: PH BB v5 val, Open Images val, and both together.
VAL_SETS = {"phbb": read_list(f"val_{PH_SOURCE}.txt"), "oi": read_list(f"val_{OI_SOURCE}.txt")}
VAL_SETS["all"] = VAL_SETS["phbb"] + VAL_SETS["oi"]
TEST_SETS = {"phbb": read_list(f"test_{PH_SOURCE}.txt"), "oi": read_list(f"test_{OI_SOURCE}.txt")}
TEST_SETS["all"] = TEST_SETS["phbb"] + TEST_SETS["oi"]
for k, v in VAL_SETS.items():
    print(f"val set {k}: {len(v)} images")
assert VAL_SETS["phbb"] and VAL_SETS["oi"], "val_<source>.txt lists missing: rebuild the dataset with the current ml.dataset"
assert not set(VAL_SETS["all"]) & set(train_list), "a validation image is in the training list"

def write_yaml(name, val_images):
    (DATA_DIR / f"{name}.txt").write_text("".join(f"{p.as_posix()}\n" for p in val_images))
    (DATA_DIR / f"{name}.yaml").write_text(
        f"path: {DATA_DIR.as_posix()}\ntrain: {json.dumps(TRAIN_ENTRIES)}\nval: {name}.txt\nnames:\n  0: license_plate\n")
    return DATA_DIR / f"{name}.yaml"

TRAIN_YAML = write_yaml("data_val_all", VAL_SETS["all"])        # Ultralytics' own validation during training
VAL_YAMLS = {k: write_yaml(f"data_val_{k}", v) for k, v in VAL_SETS.items()}

## 4. Leak check (before training)

`ml.dataset.leak_report` re-checks the built version from its files. With synthetic folders it adds them to the training side, so any synthetic image that is identical or a near duplicate of a validation or test image is counted. **All four counts must be 0.** The cell stops the run otherwise. (It checks the whole `train` split plus the synthetic folders, which covers any subset used here.)

In [ ]:
leaks = ds.leak_report(V2, extra_train=syn_folders)
print(json.dumps(leaks, indent=2))
assert all(v == 0 for v in leaks.values()), "leak found: do not train; fix the synthetic folder or the dataset first"
(RUN_DIR / "leak_report.json").write_text(json.dumps(leaks, indent=2) + "\n")
print("leak check passed (extra training folders:", [d.name for d in syn_folders] or "none", ")")

## 5. Train (resumes after a disconnect)

Same budget for the same block: epochs, size, batch and augmentation come from cell 2. Weights go straight to Drive, so `last.pt` survives a disconnect. If the session dies, run all cells again with the same `RUN_ID`: this cell finds `last.pt` and resumes. The training wall time is taken from `results.csv` (its `time` column, seconds since the run began).

In [ ]:
from ultralytics import YOLO
LAST = RUN_DIR / "train" / "weights" / "last.pt"
BEST = RUN_DIR / "train" / "weights" / "best.pt"
FINISHED = RUN_DIR / "train.finished"
if FINISHED.is_file():
    print("Training already finished for", RUN_ID)
elif LAST.is_file():
    print("Found last.pt, resuming:", LAST)
    try:
        YOLO(str(LAST)).train(resume=True)
    except Exception as e:
        if "finished" in str(e).lower() or "nothing to resume" in str(e).lower():
            print("Training had already finished:", e)
        else:
            raise
    FINISHED.write_text("done\n")
else:
    assert START != "v0" or Path(START_PATH).is_file(), f"v0 weights not found at {START_PATH}"
    YOLO(START_PATH).train(data=str(TRAIN_YAML), epochs=EPOCHS, patience=PATIENCE_USED, imgsz=IMGSZ, batch=BATCH,
                           seed=SEED, deterministic=True, device=0, project=str(RUN_DIR), name="train", exist_ok=True,
                           **AUG_ARGS)
    FINISHED.write_text("done\n")
assert BEST.is_file(), "best.pt missing"
import pandas as pd
df = pd.read_csv(RUN_DIR / "train" / "results.csv")
df.columns = [c.strip() for c in df.columns]
WALL_H = float(df["time"].iloc[-1]) / 3600 if "time" in df.columns else None
fit = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]
TRAIN_INFO = dict(epochs_run=len(df), best_epoch=int(df["epoch"].iloc[int(fit.values.argmax())]), wall_hours=WALL_H)
print("best weights:", BEST)
print(TRAIN_INFO)

## 6. Evaluation helpers

Same rules as `train_detector_v0.ipynb`, extended to several image sets:
- Each model predicts once per image at `conf` 0.01. Each prediction (highest score first) is matched to at most one unused ground-truth box at IoU 0.50. One pass serves every threshold.
- `conf`: the highest confidence that keeps validation recall at 0.95 or more; if none does, the best F1 (decision 5).
- `mask_conf`: the lowest confidence, not below 0.10, at which validation precision is at least 0.90 (decision 24).
- Both thresholds are chosen on **PH BB val + Open Images val together**. Each set (PH BB, Open Images, both) is then scored at those thresholds and reported on its own line.

In [ ]:
RESULTS = []

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def load_gt(label_path):
    if not label_path.is_file():
        return np.zeros((0, 4))
    lines = [l for l in label_path.read_text().splitlines() if l.strip()]
    if not lines:
        return np.zeros((0, 4))
    a = np.array([[float(v) for v in l.split()] for l in lines])
    cx, cy, w, h = a[:, 1], a[:, 2], a[:, 3], a[:, 4]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)

def iou_many(box, gts):
    xx1 = np.maximum(box[0], gts[:, 0]); yy1 = np.maximum(box[1], gts[:, 1])
    xx2 = np.minimum(box[2], gts[:, 2]); yy2 = np.minimum(box[3], gts[:, 3])
    inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
    a = (box[2] - box[0]) * (box[3] - box[1])
    b = (gts[:, 2] - gts[:, 0]) * (gts[:, 3] - gts[:, 1])
    return inter / np.maximum(a + b - inter, 1e-12)

def predict_images(weights, imgs, imgsz, is_pt):
    # per image: (scores, matched flags, number of ground-truth boxes)
    model = YOLO(str(weights), task="detect")
    kw = dict(imgsz=imgsz, conf=0.01, iou=0.5, verbose=False, device=0 if is_pt else "cpu")
    def chunked(size=8):
        # Ultralytics 8.4 batched every path of a long list at once (9 GiB on the GPU, then the host RAM ran out),
        # so each call gets a few images only.
        for i in range(0, len(imgs), size):
            part = imgs[i:i + size]
            yield from zip(part, model.predict([str(p) for p in part], **kw))
    per = {}
    for path, r in chunked():
        gts = load_gt(label_of(path))
        scores, matched = [], []
        if r.boxes is not None and len(r.boxes):
            sc = r.boxes.conf.cpu().numpy()
            order = np.argsort(-sc)
            sc = sc[order]; xy = r.boxes.xyxyn.cpu().numpy()[order]   # IoU is unchanged by per-axis scaling
            used = np.zeros(len(gts), dtype=bool)
            for box, s in zip(xy, sc):
                ok = False
                if len(gts):
                    ious = iou_many(box, gts)
                    ious[used] = -1
                    j = int(ious.argmax())
                    if ious[j] >= 0.5:
                        used[j] = True
                        ok = True
                scores.append(float(s)); matched.append(ok)
        per[path] = (np.array(scores), np.array(matched, dtype=bool), len(gts))
    return per

def pool(per, paths):
    s = [per[p][0] for p in paths]; m = [per[p][1] for p in paths]
    return (np.concatenate(s) if s else np.zeros(0), np.concatenate(m) if m else np.zeros(0, dtype=bool),
            sum(per[p][2] for p in paths))

def counts_at(P, t):
    scores, matched, n_gt = P
    sel = scores >= t
    tp = int((sel & matched).sum()); fp = int((sel & ~matched).sum()); fn = n_gt - tp
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / n_gt if n_gt else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(conf=round(float(t), 2), tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

def choose_threshold(P):
    rows = [counts_at(P, t) for t in np.arange(0.01, 0.99, 0.01)]
    ok = [r for r in rows if r["recall"] >= 0.95]
    if ok:
        return max(ok, key=lambda r: r["conf"]), "highest conf with recall >= 0.95"
    best = max(rows, key=lambda r: (round(r["f1"], 6), r["conf"]))
    return best, "no threshold reached recall 0.95; best F1"

def choose_mask_conf(P, floor=0.10, min_precision=0.90):
    for t in np.arange(floor, 0.99, 0.01):
        r = counts_at(P, t)
        if r["precision"] >= min_precision:
            return r
    return None

def ultralytics_map(weights, yaml_path, imgsz, is_pt):
    m = YOLO(str(weights), task="detect").val(data=str(yaml_path), split="val", imgsz=imgsz, batch=16 if is_pt else 1,
                                                device=0 if is_pt else "cpu", plots=False, verbose=False)
    return float(m.box.map50), float(m.box.map)

def evaluate(weights, imgsz, label, is_pt, sets=None, yamls=None, thresholds=None, split="val"):
    # thresholds=None: choose conf and mask_conf on the combined validation set. Otherwise use the given ones (locked comparison).
    sets = sets or VAL_SETS
    yamls = yamls or VAL_YAMLS
    per = predict_images(weights, imgs=sets["all"], imgsz=imgsz, is_pt=is_pt)
    Pall = pool(per, sets["all"])
    if thresholds is None:
        v, rule = choose_threshold(Pall)
        mk = choose_mask_conf(Pall)
        conf, mask_conf = v["conf"], (mk["conf"] if mk else None)
    else:
        conf, mask_conf, rule = thresholds["conf"], thresholds["mask_conf"], "from MODEL.json"
    out = dict(model=label, split=split, imgsz=imgsz, conf=conf, mask_conf=mask_conf, rule=rule, sets={})
    for name in ("phbb", "oi", "all"):
        P = pool(per, sets[name])
        at_conf = counts_at(P, conf)
        at_mask = counts_at(P, mask_conf) if mask_conf is not None else None
        map50, map5095 = ultralytics_map(weights, yamls[name], imgsz, is_pt)
        out["sets"][name] = dict(images=len(sets[name]), gt_boxes=P[2], at_conf=at_conf, at_mask=at_mask, map50=map50, map50_95=map5095)
    RESULTS.append(out)
    print(f"[{label} @ {imgsz}, {split}] conf={conf} ({rule}), mask_conf={mask_conf}")
    for name, s in out["sets"].items():
        c, m = s["at_conf"], s["at_mask"]
        line = f"  {name:5} n={s['images']:5} boxes={s['gt_boxes']:5} | at conf: P={c['precision']:.4f} R={c['recall']:.4f}"
        if m:
            line += f" | at mask_conf: P={m['precision']:.4f} R={m['recall']:.4f} (TP={m['tp']} FP={m['fp']} FN={m['fn']})"
        print(line + f" | mAP50={s['map50']:.4f} mAP50-95={s['map50_95']:.4f}")
    return out

def onnx_shapes(path):
    import onnxruntime as ort
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    return list(sess.get_inputs()[0].shape), [list(o.shape) for o in sess.get_outputs()]

## 7. Validation scores of `best.pt`, then ONNX export at 640 and 416

For each export the ONNX file is scored on the same validation sets and `conf` and `mask_conf` are chosen again on that file. `MODEL.json` is written next to it with the file name, sha256, thresholds and version, plus the ONNX **input and output shapes**. YOLOv8 and YOLO11 give an output of shape `(1, 5, N)`; YOLO26 exports end-to-end without NMS and gives another shape, so the shape is recorded, not assumed. The web app reads `(1, 5, N)` today, so a model with another shape needs app work before it can be served.

In [ ]:
RESULTS.clear()
import gc
gc.collect(); torch.cuda.empty_cache()          # training leaves GPU memory behind; free it before scoring
try:
    res_pt = evaluate(BEST, IMGSZ, f"{RUN_ID} best.pt", is_pt=True)
except torch.cuda.OutOfMemoryError as e:
    # Seen on a free T4 after training in the same session. The ONNX files below are scored on the CPU instead.
    print("best.pt scoring on the GPU ran out of memory and was skipped:", str(e)[:120])
    res_pt = None
    gc.collect(); torch.cuda.empty_cache()
SUMMARY = dict(settings=SETTINGS, train=TRAIN_INFO, leak_report=leaks, synthetic_counts=syn_count,
               training_lines=len(train_list), pt=res_pt, onnx={})

In [ ]:
REL = RELEASE / RUN_ID
EXPORT_SIZES = sorted({640, 416, IMGSZ}, reverse=True)
for s in EXPORT_SIZES:
    d = REL / f"plate-{s}"
    d.mkdir(parents=True, exist_ok=True)
    exported = YOLO(str(BEST)).export(format="onnx", imgsz=s, opset=12, simplify=True, dynamic=False, half=False, nms=False)
    dst = d / f"plate_{s}.onnx"
    shutil.copy(exported, dst)
    in_shape, out_shapes = onnx_shapes(dst)
    print(f"{dst.name}: input {in_shape}, outputs {out_shapes}")
    res = evaluate(dst, s, f"{RUN_ID} onnx", is_pt=False)
    meta = dict(file=dst.name, sha256=sha256_file(dst), imgsz=s, conf=res["conf"], mask_conf=res["mask_conf"], iou=0.5,
                names=["license_plate"], version=f"{RUN_ID}-{s}", input_shape=in_shape, output_shapes=out_shapes,
                run_id=RUN_ID, start=START, seed=SEED)
    (d / "MODEL.json").write_text(json.dumps(meta, indent=2) + "\n")
    SUMMARY["onnx"][str(s)] = dict(result=res, model_json=meta)
    print("  wrote", d / "MODEL.json")
shutil.copy(BEST, REL / "best.pt")
shutil.copy(RUN_DIR / "train" / "results.csv", REL / "results.csv")
(RUN_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=float) + "\n")
print("saved", RUN_DIR / "summary.json")

## 8. Summary of this run

One Markdown table of the measured numbers above. Bring this cell's output back, or `summary.json` from the run folder on Drive.

In [ ]:
def f4(x):
    return "n/a" if x is None else f"{x:.4f}"

print(f"Run {RUN_ID}: start {START}, seed {SEED}, {EPOCHS} epochs at {IMGSZ}, aug {AUG}, patience {PATIENCE_USED}")
print(f"epochs run {TRAIN_INFO['epochs_run']}, best epoch {TRAIN_INFO['best_epoch']} (as numbered in results.csv), "
      f"training wall time {f4(TRAIN_INFO['wall_hours'])} h (from results.csv)")
print(f"sources: {SOURCES} minus [{EXCLUDE_SOURCES}]; caps: {CAPS or 'none'}; synthetic: {SYNTHETIC or 'none'} {syn_count}; PH BB listed x{PH_OVERSAMPLE}")
if STARTED_FROM_V0:
    print("CAVEAT: started from v0: PH BB rows are optimistic (v0 saw part of PH BB val and test); Open Images rows are clean.")
print()
print("| model | imgsz | set | images | boxes | conf | mask_conf | P@mask | R@mask | P@conf | R@conf | mAP@0.50 | mAP@0.50:0.95 |")
print("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
for r in RESULTS:
    for name, s in r["sets"].items():
        m = s["at_mask"]
        print(f"| {r['model']} | {r['imgsz']} | {name} | {s['images']} | {s['gt_boxes']} | {r['conf']} | {r['mask_conf']} | "
              f"{f4(m['precision'] if m else None)} | {f4(m['recall'] if m else None)} | {f4(s['at_conf']['precision'])} | "
              f"{f4(s['at_conf']['recall'])} | {f4(s['map50'])} | {f4(s['map50_95'])} |")
print()
print("ONNX files (sha256 from MODEL.json)")
for s, v in SUMMARY["onnx"].items():
    m = v["model_json"]
    print(f"- plate-{s}: {m['file']} {m['sha256']} conf={m['conf']} mask_conf={m['mask_conf']} "
          f"input {m['input_shape']} outputs {m['output_shapes']}")

## 9. Collect all runs (optional)

Reads every `summary.json` under `runs/compare/` on Drive and prints one row per run at the 640 ONNX file, for the comparison table. Seeds are separate runs (`C1-s0`, `C1-s1`, ...): report one row per seed, then the mean and spread. Run this cell alone after the others have finished; it needs only cell 1.

In [ ]:
rows = []
for f in sorted(RUNS.glob("*/summary.json")):
    s = json.loads(f.read_text())
    o = s["onnx"].get("640")
    if not o:
        continue
    r = o["result"]
    st = s["settings"]
    rows.append((st["run_id"], st["start"], st["seed"], r["mask_conf"], r["sets"]["all"], r["sets"]["oi"], r["sets"]["phbb"],
                 s["train"].get("wall_hours"), o["model_json"]["output_shapes"], o["model_json"]["sha256"][:12], st.get("started_from_v0")))
def g(x, k):
    m = x["at_mask"]
    return "n/a" if not m else f"{m[k]:.4f}"
print("| run | start | seed | mask_conf | all R@mask | all P@mask | OI R@mask | PH BB R@mask | wall h | ONNX output | sha256[:12] |")
print("|---|---|---|---|---|---|---|---|---|---|---|")
for run, start, seed, mc, a, oi, ph, wall, shp, sha, v0 in rows:
    ph_txt = g(ph, "recall") + (" (v0 start: caveat)" if v0 else "")
    wall_txt = "n/a" if wall is None else f"{wall:.3f}"
    print(f"| {run} | {start} | {seed} | {mc} | {g(a, 'recall')} | {g(a, 'precision')} | {g(oi, 'recall')} | {ph_txt} | {wall_txt} | {shp} | {sha} |")

## 10. Number formats (block D4, export only, optional)

Exports the model in 32-bit and 16-bit, scores each on validation, and prints the file sizes. Time each file on the phone with the app's Benchmark. 16-bit has been reported to break WebGPU (`docs/PLAN.md` section 5). 8-bit uses ONNX Runtime's dynamic quantization (`QUInt8`): a quick option, not a calibrated one, and not tested yet, so look at the validation score before trusting it. Set `FORMAT_RUN_ID` to the finished run to export.

In [ ]:
RUN_FORMATS = False       #@param {type:"boolean"}
FORMAT_RUN_ID = "C1-s0"   #@param {type:"string"}
FORMAT_IMGSZ = 640        #@param [320, 416, 640] {type:"raw"}
# Off by default, so "Run all" for a training run skips it. Switch on and run this cell alone.
if not RUN_FORMATS:
    print("Number-format export skipped (RUN_FORMATS is off).")
else:
    fbest = RUNS / FORMAT_RUN_ID / "train" / "weights" / "best.pt"
    assert fbest.is_file(), f"{fbest} not found"
    fdir = RELEASE / FORMAT_RUN_ID / f"formats-{FORMAT_IMGSZ}"
    fdir.mkdir(parents=True, exist_ok=True)
    RESULTS.clear()
    fp32 = Path(YOLO(str(fbest)).export(format="onnx", imgsz=FORMAT_IMGSZ, opset=12, simplify=True, half=False, nms=False))
    shutil.copy(fp32, fdir / "fp32.onnx")
    fp16 = Path(YOLO(str(fbest)).export(format="onnx", imgsz=FORMAT_IMGSZ, opset=12, simplify=True, half=True, nms=False, device=0))
    shutil.copy(fp16, fdir / "fp16.onnx")
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(str(fdir / "fp32.onnx"), str(fdir / "int8.onnx"), weight_type=QuantType.QUInt8)
    for tag in ("fp32", "fp16", "int8"):
        p = fdir / f"{tag}.onnx"
        print(f"{tag}: {p.stat().st_size / 1e6:.2f} MB, shapes {onnx_shapes(p)}")
        evaluate(p, FORMAT_IMGSZ, f"{FORMAT_RUN_ID} {tag}", is_pt=False)

## 11. LOCKED COMPARISON, RUN ONCE (block E)

**Do not run this cell while choosing anything.** It scores models on the **test** splits (PH BB v5 test and Open Images test), at thresholds read from each model's `MODEL.json` (chosen on validation). Run it after the choice is locked and written down. It writes a marker file on Drive and refuses to run a second time.

It scores `v0` (its release folder from `train_detector_v0.ipynb`) and the chosen run. v0's PH BB test score carries the leak caveat (v0 saw part of that split in training).

In [ ]:
LOCK_CONFIRM = False   #@param {type:"boolean"}
LOCKED_RUN_ID = "C1-s0"   #@param {type:"string"}
LOCKED_IMGSZ = 640        #@param [416, 640] {type:"raw"}
MARKER = ROOT / "locked_comparison_done.json"
if not LOCK_CONFIRM:
    # "Run all" during the study ends here without touching the test splits.
    print("Locked comparison not run (LOCK_CONFIRM is off). The test splits were not read.")
else:
    assert not MARKER.exists(), f"The locked comparison already ran ({MARKER}). It is scored once."
    models = {
        "v0": ROOT / "release" / "yolov8n" / f"plate-{LOCKED_IMGSZ}",
        LOCKED_RUN_ID: RELEASE / LOCKED_RUN_ID / f"plate-{LOCKED_IMGSZ}",
    }
    RESULTS.clear()
    locked = {}
    test_yamls = {k: write_yaml(f"data_test_{k}", v) for k, v in TEST_SETS.items()}   # val: points at the test list
    for label, d in models.items():
        meta = json.loads((d / "MODEL.json").read_text())
        onnx_path = d / meta["file"]
        assert sha256_file(onnx_path) == meta["sha256"], f"{label}: ONNX file does not match MODEL.json"
        locked[label] = evaluate(onnx_path, LOCKED_IMGSZ, f"{label} onnx", is_pt=False, sets=TEST_SETS, yamls=test_yamls,
                                 thresholds=dict(conf=meta["conf"], mask_conf=meta["mask_conf"]), split="test")
    MARKER.write_text(json.dumps(locked, indent=2, default=float) + "\n")
    print("locked comparison saved to", MARKER)
    print("v0 caveat: its PH BB test score is optimistic (v0 trained on part of that split). Open Images test is clean for every model.")